In [3]:
"""
nivht_batch_process.py

Optimized batch processing for large image dataset (e.g., 5000 images).
Computes edge-methods metrics and produces boxplots comparing methods,
including the Proposed NIVHT method vs classical filters.

Dependencies:
    pip install opencv-python numpy pandas scikit-image matplotlib tqdm joblib
"""

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from joblib import Parallel, delayed
from skimage import filters, img_as_ubyte
from skimage.morphology import skeletonize, disk
from skimage.filters.rank import entropy as rank_entropy
from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import mean_squared_error as sk_mse
from skimage.metrics import structural_similarity as sk_ssim
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import math

# ---------------------------
# CONFIG
# ---------------------------
INPUT_FOLDER = r"C:/programming/pythonML/computerVision/Breast_cancer/5000/dataset"
OUTPUT_DIR = r"C:/programming/pythonML/computerVision/Breast_cancer/5000/processed"
OUTPUT_GRAPHS = os.path.join(OUTPUT_DIR, "graphs")
OUTPUT_EDGE_IMAGES = os.path.join(OUTPUT_DIR, "edge_variations")  # optional save
CSV_OUT = os.path.join(OUTPUT_DIR, "metrics_summary.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_GRAPHS, exist_ok=True)
os.makedirs(OUTPUT_EDGE_IMAGES, exist_ok=True)

TARGET_SIZE = (150, 200)  # (width, height)
N_JOBS = max(1, os.cpu_count() - 1)

METHOD_LABELS = ["Proposed_NIVHT", "Edge_Thinning_NIVHT", "Roberts", "Sobel", "Canny", "Prewitt", "Laplacian"]

# ---------------------------
# UTILITIES / METRICS
# ---------------------------
def calculate_epi(orig_img_uint8, edge_bin):
    if edge_bin.sum() == 0:
        return 0.0
    orig = orig_img_uint8.astype(np.float32)
    mask = edge_bin > 0
    mean_edge_intensity = orig[mask].mean()
    overall_mean = orig.mean() + 1e-8
    return float(mean_edge_intensity / overall_mean)

def compute_entropy(edge_bin_uint8):
    try:
        ent_img = rank_entropy(edge_bin_uint8, disk(3))
        return float(ent_img.mean())
    except Exception:
        hist = np.bincount(edge_bin_uint8.flatten(), minlength=256).astype(np.float64)
        prob = hist / (hist.sum() + 1e-12)
        prob = prob[prob > 0]
        return float(-(prob * np.log2(prob)).sum())

def hough_line_lengths(edge_bin):
    lines = cv2.HoughLinesP(edge_bin, 1, np.pi / 180, threshold=50, minLineLength=8, maxLineGap=5)
    if lines is None:
        return 0.0, 0.0, 0.0
    lengths = [math.hypot(x2-x1, y2-y1) for x1, y1, x2, y2 in lines[:,0]]
    return float(np.mean(lengths)), float(np.max(lengths)), float(np.min(lengths))

def connected_components_count(edge_bin):
    if edge_bin.dtype != np.uint8:
        edge_bin = (edge_bin > 0).astype(np.uint8) * 255
    num_labels, _ = cv2.connectedComponents(edge_bin)
    return int(max(0, num_labels - 1))

def safe_psnr_mse(img_ref, img_test):
    try:
        psnr_v = sk_psnr(img_ref, img_test, data_range=255)
    except Exception:
        psnr_v = sk_psnr(img_ref.astype(np.uint8), (img_test*255).astype(np.uint8), data_range=255)
    try:
        mse_v = sk_mse(img_ref, img_test)
    except Exception:
        mse_v = float(np.mean((img_ref.astype(np.float32) - img_test.astype(np.float32))**2))
    return float(psnr_v), float(mse_v)

# ---------------------------
# PER-IMAGE PROCESSING
# ---------------------------
def process_single_image(filepath, save_edge_images=False):
    out_rows = []
    filename = os.path.basename(filepath)
    img_bgr = cv2.imread(filepath, cv2.IMREAD_COLOR)
    if img_bgr is None:
        return []
    img_resized = cv2.resize(img_bgr, TARGET_SIZE, interpolation=cv2.INTER_AREA)
    img_gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
    img_inv = cv2.bitwise_not(img_gray)
    img_ref = img_inv

    roberts_edges = filters.roberts(img_inv)
    sobel_edges = filters.sobel(img_inv)
    prewitt_edges = filters.prewitt(img_inv)
    canny_edges = cv2.Canny(img_inv, 50, 150)
    laplacian = cv2.Laplacian(img_inv, cv2.CV_64F)
    laplacian_edges = np.uint8(np.absolute(laplacian))

    img_eq = cv2.equalizeHist(img_inv)
    _, high_intensity = cv2.threshold(img_eq, 100, 255, cv2.THRESH_BINARY)
    edges_canny = cv2.Canny(img_inv, 50, 200)
    combined_edges = cv2.bitwise_and(edges_canny, high_intensity)
    skeleton = skeletonize(combined_edges > 0)
    thin_edges = img_as_ubyte(skeleton)

    skeleton_canny = skeletonize((edges_canny > 0))
    thin_edges_only = img_as_ubyte(skeleton_canny)

    method_images = {
        "Proposed_NIVHT": thin_edges,
        "Edge_Thinning_NIVHT": thin_edges_only,
        "Roberts": np.uint8(np.clip(roberts_edges * 255, 0, 255)),
        "Sobel": np.uint8(np.clip(sobel_edges * 255, 0, 255)),
        "Canny": np.uint8(canny_edges),
        "Prewitt": np.uint8(np.clip(prewitt_edges * 255, 0, 255)),
        "Laplacian": np.uint8(laplacian_edges)
    }

    if save_edge_images:
        base = os.path.splitext(filename)[0]
        for name, im in method_images.items():
            out_path = os.path.join(OUTPUT_EDGE_IMAGES, f"{base}_{name}.png")
            cv2.imwrite(out_path, im)

    for name, edge_img in method_images.items():
        edge_uint8 = np.uint8(edge_img)
        edge_bin = (edge_uint8 > 0).astype(np.uint8) * 255
        num_edges = int(np.count_nonzero(edge_bin))
        edge_density = float(num_edges) / float(edge_bin.size + 1e-12)
        epi = calculate_epi(img_ref, edge_bin)
        ent = compute_entropy(edge_uint8)
        psnr_v, mse_v = safe_psnr_mse(img_ref, edge_uint8)
        avg_len, max_len, min_len = hough_line_lengths(edge_bin)
        conn_comp = connected_components_count(edge_bin)

        out_rows.append({
            "Image": filename,
            "Method": name,
            "Number_of_Edges": num_edges,
            "Edge_Density": edge_density,
            "EPI": epi,
            "Entropy": ent,
            "PSNR": psnr_v,
            "MSE": mse_v,
            "Avg_Edge_Length": avg_len,
            "Max_Edge_Length": max_len,
            "Min_Edge_Length": min_len,
            "Connected_Components": conn_comp
        })
    return out_rows

# ---------------------------
# BATCH PROCESSING
# ---------------------------
def batch_process(input_folder=INPUT_FOLDER, n_jobs=N_JOBS, save_edge_images=False, limit=None):
    files = [os.path.join(input_folder, f) for f in os.listdir(input_folder)
             if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif'))]
    files.sort()
    if limit:
        files = files[:limit]
    if len(files) == 0:
        raise FileNotFoundError(f"No images found in {input_folder}")

    results = []
    with Parallel(n_jobs=n_jobs, backend="loky") as pool:
        tasks = pool(delayed(process_single_image)(fp, save_edge_images) for fp in files)
        for res in tqdm(tasks, total=len(files), desc="Images processed"):
            if res:
                results.extend(res)

    df = pd.DataFrame(results)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    df.to_csv(CSV_OUT, index=False)
    print(f"Saved metrics to: {CSV_OUT}")
    return df

# ---------------------------
# PLOTTING
# ---------------------------
def plot_boxplots(df, output_dir=OUTPUT_GRAPHS):
    os.makedirs(output_dir, exist_ok=True)
    metrics = ["Number_of_Edges", "Edge_Density", "EPI", "Entropy", "PSNR", "MSE", "Avg_Edge_Length", "Connected_Components"]
    df["Method"] = pd.Categorical(df["Method"], categories=METHOD_LABELS, ordered=True)

    for metric in metrics:
        plt.figure(figsize=(10, 6))
        df.boxplot(column=metric, by="Method", grid=False)
        plt.title(f"{metric} — comparison across methods")
        plt.suptitle("")
        plt.ylabel(metric)
        plt.tight_layout()
        out_path = os.path.join(output_dir, f"boxplot_{metric}.png")
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved boxplot: {out_path}")

def plot_barplots(df, output_dir=OUTPUT_GRAPHS):
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for metric in numeric_cols:
        means = df.groupby("Method")[metric].mean()
        stds = df.groupby("Method")[metric].std()
        plt.figure(figsize=(10, 6))
        plt.bar(means.index, means.values, yerr=stds.values, capsize=5)
        plt.ylabel(metric)
        plt.title(f"{metric} — Mean ± Std across methods")
        plt.xticks(rotation=30)
        plt.tight_layout()
        out_path = os.path.join(output_dir, f"barplot_{metric}.png")
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved bar plot: {out_path}")

# ---------------------------
# MAIN
# ---------------------------
if __name__ == "__main__":
    TEST_LIMIT = None  # Set to e.g., 200 for testing
    df_metrics = batch_process(input_folder=INPUT_FOLDER, n_jobs=N_JOBS, save_edge_images=False, limit=TEST_LIMIT)

    plot_boxplots(df_metrics, output_dir=OUTPUT_GRAPHS)
    plot_barplots(df_metrics, output_dir=OUTPUT_GRAPHS)

    # Summary stats (median, mean, std)
    numeric_cols = df_metrics.select_dtypes(include=[np.number]).columns
    summary_median = df_metrics.groupby("Method")[numeric_cols].median().add_suffix("_Median")
    summary_mean   = df_metrics.groupby("Method")[numeric_cols].mean().add_suffix("_Mean")
    summary_std    = df_metrics.groupby("Method")[numeric_cols].std().add_suffix("_Std")
    summary = pd.concat([summary_median, summary_mean, summary_std], axis=1).reset_index()

    summary_path = os.path.join(OUTPUT_DIR, "metrics_summary_by_method.csv")
    summary.to_csv(summary_path, index=False)
    print(f"Summary (median, mean, std) saved to: {summary_path}")
    print("Done.")


Images processed: 100%|██████████| 7632/7632 [00:00<00:00, 1526729.06it/s]


Saved metrics to: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\metrics_summary.csv
Saved boxplot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\boxplot_Number_of_Edges.png
Saved boxplot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\boxplot_Edge_Density.png
Saved boxplot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\boxplot_EPI.png
Saved boxplot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\boxplot_Entropy.png
Saved boxplot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\boxplot_PSNR.png
Saved boxplot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\boxplot_MSE.png
Saved boxplot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\boxplot_Avg_Edge_Length.png
Saved boxplot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\boxplot_Connected_Compon

C:\Users\Administrator\AppData\Local\Temp\ipykernel_5788\4181818934.py:217: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  means = df.groupby("Method")[metric].mean()
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5788\4181818934.py:218: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stds = df.groupby("Method")[metric].std()


Saved bar plot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\barplot_Number_of_Edges.png
Saved bar plot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\barplot_Edge_Density.png
Saved bar plot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\barplot_EPI.png
Saved bar plot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\barplot_Entropy.png
Saved bar plot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\barplot_PSNR.png
Saved bar plot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\barplot_MSE.png
Saved bar plot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\barplot_Avg_Edge_Length.png
Saved bar plot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\barplot_Max_Edge_Length.png
Saved bar plot: C:/programming/pythonML/computerVision/Breast_cancer/5000/processed\graphs\bar

C:\Users\Administrator\AppData\Local\Temp\ipykernel_5788\4181818934.py:242: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_median = df_metrics.groupby("Method")[numeric_cols].median().add_suffix("_Median")
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5788\4181818934.py:243: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_mean   = df_metrics.groupby("Method")[numeric_cols].mean().add_suffix("_Mean")
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5788\4181818934.py:244: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. 

...

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [5]:
# ------------------------------------------------------
# Boxplot for Average, Min, and Max Edge Length
# ------------------------------------------------------
import seaborn as sns

# Pick only the required metrics
edge_metrics = ["Average_Edge_Length", "Min_Edge_Length", "Max_Edge_Length"]

# Melt dataframe into long format
df_edges = df_metrics.melt(id_vars=["Method"], 
                           value_vars=edge_metrics, 
                           var_name="Metric", 
                           value_name="Value")

# Plot combined boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(x="Metric", y="Value", hue="Method", data=df_edges)

plt.title("Boxplot of Edge Length Metrics Across Methods", fontsize=14, weight="bold")
plt.ylabel("Edge Length Value")
plt.xlabel("Metric")
plt.xticks(rotation=0)
plt.legend(title="Method", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

# Save output
out_path = os.path.join(OUTPUT_GRAPHS, "boxplot_edge_lengths.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close()
print(f"Saved boxplot of edge length metrics: {out_path}")


KeyError: "The following id_vars or value_vars are not present in the DataFrame: ['Average_Edge_Length']"